# The Bowman-2018 pipeline: single-day analysis

This notebook performs the single-day parts of the analysis pipeline that matches the Bowman+2018 results, using the python `edges-collab` stack. This loads in a full day of data, flags out bad integrations, flags RFI, and averages all retained LSTs.

In [ ]:
datadir: str = "/data5/edges/data/2014_February_Boolardy/mro/low/"
alan_output_dir: str = "/home/smurray/data4/edges/alans-pipeline/scripts/H2CaseFieldData/"
outdir: str = "."

year: int = 2016
day: int = 250

do_aux_filter: bool = True
aux_filter_params: dict = {
    'minima': {},
    'maxima': {
        'adcmax': 0.35
    }
}

lst_min: float = 6.0
lst_max: float = 18.0
use_alan_coordinates: bool = True

do_power_percent_filter: bool = True
power_percent_params = {
    'min_threshold': 0.7,
    'max_threshold': 3
}

do_peak_orbcomm_filter: bool = True
peak_orbcomm_params = {
    'threshold': 40.0
}

do_maxfm_filter: bool = True
maxfm_filter_params = {
    'threshold': 200.0
}

do_rmsf_filter: bool = True
rmsf_filter_params = {
    'freq_range': [60., 80.],
    'threshold': 200.0
}

gauss_smooth_params = {
    "size": 8,
    'decimate_at': 0,
    'maintain_flags': 1,
    'flag_threshold': 0.25,
    'use_nsamples': True,
}

do_balun_connector_loss: bool = True
do_antenna_loss: bool = False
do_ground_loss: bool = False
do_beam_correction: bool = False  # should be True to match B18      

first_freqcut_min: float = 40.0  # MHz
first_freqcut_max: float = 100.0  # MHz

inject_flags: bool = True
calfile: str = "/data4/smurray/edges/alans-pipeline/edges-cal-outputs/specal.txt"
#calfile: str = "/data4/smurray/edges/alans-pipeline/scripts/H2Case-fittpfix//specal.txt"

In [ ]:
year = int(year)
day = int(day)

## Imports and Versions

In [ ]:
from pathlib import Path
from importlib.metadata import version
from copy import deepcopy
import sys 

import matplotlib.pyplot as plt
import numpy as np
from astropy import units as un
from astropy.coordinates import Longitude
import pandas as pd
from pygsdata import plots, GSData, GSFlag
from pygsdata.select import select_freqs, select_lsts, lst_selector

from read_acq.gsdata import read_acq_to_gsdata, fast_lst_setter

from edges_cal import Calibrator
from edges_cal.alanmode import read_specal_as_calibrator, read_spec_txt
from edges_cal import modelling as mdl
from edges_cal.tools import dicke_calibration

from edges_analysis.calibration.calibrate import approximate_temperature
from edges_analysis.const import KNOWN_TELESCOPES
from edges_analysis.averaging.freqbin import gauss_smooth
from edges_analysis.averaging import average_over_times, NsamplesStrategy
from edges_analysis.filters import filters
from pygsdata.coordinates import gha2lst
from edges_pipelines import utils


In [ ]:
utils.print_versions()

## Load Data

In [ ]:
cache = Path(outdir)
cache.mkdir(exist_ok=True, parents=True)

In [ ]:
alan_output_dir = Path(alan_output_dir) / str(utils.yday_to_alanday(year, day))

In [ ]:
obsname = f"{year}-{day:>03}"

In [ ]:
print(f"In this notebook, we are averaging data on day {obsname}, corresponding to Alan's numbering: {alan_output_dir.name}")

In [ ]:
files_to_load = sorted((Path(datadir) / str(year)).glob(f"{year}_{day:03}_*.acq"))
data = read_acq_to_gsdata(files_to_load, telescope=KNOWN_TELESCOPES['edges-low-alan'], name=obsname, lst_setter=fast_lst_setter)

In [ ]:
print(f"Number of integrations in the data: {data.ntimes}")

In [ ]:
if calfile.endswith(".txt"):
    calobs = read_specal_as_calibrator(calfile, t_load=300, t_load_ns=1000)
else:
    calobs = Calibrator.from_calfile(calfile)

## Multi-Integration Data Workflow

### Select LSTs

In [ ]:
mask = lst_selector(data.lsts, data.loads, load='all', lst_range=[lst_min, lst_max], gha=True)

In [ ]:
time_flags_alan = np.genfromtxt(alan_output_dir/"time_flags.txt", names=True)

In [ ]:
if len(time_flags_alan) == len(mask)+1:
    # Very rarely, an ACQ spectrum file is truncated on the last line (i.e. does not have all its spectra)
    # In this case, read_acq discards the line (and full 3-position switch cycle) entirely, while the C-code
    # reads it in and creates a time-flag entry. We ensure that this extra cycle is indeed flagged, and then
    # ignore it
    any_flagged = any(time_flags_alan[-1][name]<0 for name in time_flags_alan.dtype.names if not name.startswith("HA"))
    assert any_flagged
    
    time_flags_alan = time_flags_alan[:-1]

In [ ]:
alan_gha_mask = time_flags_alan['gha0'].astype(bool) & time_flags_alan['gha1'].astype(bool) & time_flags_alan['gha2'].astype(bool)

In [ ]:
# Check that our selected LSTs exactly match Alan's
assert np.sum(alan_gha_mask ^ mask) == 0

In [ ]:
data = select_lsts(data, lst_range=[lst_min, lst_max], gha=True, load='all')  # need to re-implement use_alan_coordinates...

In [ ]:
if data.ntimes == 0:
    print("There are no LSTs in the range on this day! Exiting")
    if np.any(alan_gha_mask):
        raise AssertionError("THe C-code picked up some LSTs, but this notebook didn't!")
    else:
        sys.exit(0)

In [ ]:
print(data.lsts.min(), data.lsts.max())

In [ ]:
print(f"New minimum and maximum GHA: {data.gha.min()} -- {data.gha.max()}")

In [ ]:
print(f"New number of integrations: {data.ntimes}")

### Flag out bad integrations

In [ ]:
if do_aux_filter:
    data = filters.aux_filter(data, **aux_filter_params)

In [ ]:
if do_power_percent_filter:
    data = filters.power_percent_filter(data, **power_percent_params)

### Dicke-Switch Calibration and Approximating Temperature

In [ ]:
data = dicke_calibration(data)

In [ ]:
data = approximate_temperature(data, tload=calobs.t_load, tns=calobs.t_load_ns)

### More Integration Filters

In [ ]:
if do_peak_orbcomm_filter:
    data = filters.peak_orbcomm_filter(data, **peak_orbcomm_params)

In [ ]:
if do_maxfm_filter:
    data = filters.maxfm_filter(data, **maxfm_filter_params)

In [ ]:
if do_rmsf_filter:
    data = filters.rmsf_filter(data, **rmsf_filter_params)

In [ ]:
plots.plot_waterfall(data, vmin= 0, vmax=1.2e4);

In [ ]:
# Get the number of flags in Alan's file for each filter
tf_alan = time_flags_alan[alan_gha_mask]

aux_mask = tf_alan['adc0'].astype(bool) & tf_alan['adc1'].astype(bool) & tf_alan['adc2'].astype(bool)
alan_flags = {
    'aux': ~aux_mask,
    'power_percent': tf_alan['ppercent'][aux_mask] <= 0,
    'peak_orbcomm': tf_alan['pkpower'][aux_mask] <= 0,
    'maxfm': tf_alan['fmpwr'][aux_mask] <= 0,
    'rmsf': tf_alan['rmsf'][aux_mask] <= 0,
}

In [ ]:
table = []
for name, flg in data.flags.items():
    key = name.replace("_filter", "")
    ea_flags = flg.full_rank_flags[0,0,:,0]
    
    entry = {
        'name': key, 
        '# flags': np.sum(ea_flags), 
        '# flags (C)': np.sum(alan_flags[key]),
        '# differing flags': np.sum(ea_flags[aux_mask] ^ alan_flags[key]) if key!='aux' else np.sum(ea_flags ^ alan_flags[key]),
        "% flagged": 100*np.sum(flg.full_rank_flags[0,0,:,0])/flg.full_rank_flags.shape[2]
    }
    table.append(entry)
    
ea_complete = data.complete_flags[0,0,:, 0]
ccode_complete = np.any([tf_alan[key] <= 0 for key in ['ppercent', 'pkpower', 'd150', 'dloadmax', 'rmsf', 'fmpwr']], axis=0)

table.append({
    "name": "TOTAL (w/ overlapping)", 
    "# flags": np.sum(ea_complete), 
    "% flagged": 100*np.sum(ea_complete)/data.ntimes,
    "# flags (C)": np.sum(ccode_complete),
    "# differing flags": np.sum(ea_complete ^ ccode_complete),
    })
table = pd.DataFrame(table)
table = table.set_index('name')

def highlight_differing(s):
    return np.where(s > 0, 'background-color:orange;', 'background-color:green;')

slice_ = ["# differing flags"]
table.style.apply(highlight_differing, axis=0, subset=slice_)

In [ ]:
# Actually create an error if the total differing flags is non-zero
assert table.loc['TOTAL (w/ overlapping)']['# differing flags']==0

In [ ]:
# Exit early if there's no data left.
if np.sum(~data.complete_flags)==0:
    sys.exit(0)

In [ ]:
# Save the flags
np.savez(cache / f"{obsname}.integration-flags.npz", **{name: data.flags[name].flags for name in data.flags})

### Average Over LSTs

In [ ]:
# Save out a list of LSTs used in the data (unflagged) in case we want to use these for later beam-factor calculations.
np.savetxt(f"{obsname}.lsts.txt", data.lsts.hourangle)

In [ ]:
lstbin_data = average_over_times(
    data, use_resids=False, reference_lst=gha2lst(Longitude(0.5*(lst_max + lst_min)*un.hourangle)), nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES_UNIFORM
)

In [ ]:
data = select_freqs(lstbin_data, freq_range=[first_freqcut_min*un.MHz, first_freqcut_max*un.MHz])
pre_rfi_data = deepcopy(data)

In [ ]:
utils.plot_single_spectrum(data)

## Single-LST-Bin Workflow

### RFI Excision

In [ ]:
pre_rfi_file = (cache/ obsname).with_suffix('.pre-rfi.gsh5')
try:
    pre_rfi_data.write_gsh5(pre_rfi_file);
except NameError:
    pre_rfi_data = GSData.from_file(pre_rfi_file)

In [ ]:
alan_rfi_flags = ~np.genfromtxt(alan_output_dir/"rfi_flags.txt").astype(bool)

In [ ]:
if inject_flags:
    rfiflg = GSFlag(alan_rfi_flags[-1], axes=('freq',))
    data = pre_rfi_data.add_flags('injected-rfi', rfiflg, append_to_file=False)
else:
    data = filters.rfi_model_nonlinear_window_filter(
        pre_rfi_data,
        max_iter=100,
        model = mdl.Fourier(n_terms=37, period=1.5, transform=mdl.ZerotooneTransform(
            range=(
                data.freqs.min().to_value("MHz"), 
                data.freqs.max().to_value("MHz"),
            )
        )),
        fit_kwargs = {"method": 'alan-qrd'},
        window_frac = 16,
        min_window_size = 10,
        threshold = 2.5,
        reflag_thresh = 1.0,
        watershed = {
            1.0: 4,
            10.0: 8,
            100.0: 16
        }
    )

In [ ]:
print(f"{data.complete_flags.sum()}/{data.nfreqs} channels flagged during xrfi")

In [ ]:
utils.plot_single_spectrum(data)

### Smooth Over Frequency

In [ ]:
pre_smooth_data = deepcopy(data)

In [ ]:
data = gauss_smooth(data, **gauss_smooth_params)

In [ ]:
# After smoothing, any flags over the frequency axis are removed, but nsamples is maintained correctly.
ourflags = data.nsamples[0,0,0] == 0

In [ ]:
pre_cal_data = deepcopy(data)

In [ ]:
alan_avg, n = read_spec_txt(alan_output_dir/"spegva.txt")

In [ ]:
ndiff_flags =  np.sum(~ourflags ^ (alan_avg['weight']>0))
if inject_flags:
    assert ndiff_flags == 0
else:
    print("Number of differing flags: ", ndiff_flags)
    print("Number of flags in edges-analysis: ", np.sum(ourflags))
    print("Number from Alan's code: ", np.sum(alan_avg['weight']==0))

In [ ]:
utils.plot_single_spectrum(pre_cal_data, alan_avg['spectra'])

In [ ]:
cache / f"{obsname}.averaged.gsh5"

In [ ]:
pre_cal_data.write_gsh5(cache/f"{obsname}.averaged.gsh5");